
# KAMP ④ X-ray 이물질 탐지 — Full Colab Notebook

**브라우저에서 1.7GB급 ZIP을 직접 업로드**하고 아래를 자동 수행합니다.

1. 대용량 ZIP chunk upload
2. 압축 해제 / 파일 구조 감사
3. 이미지 건전성 점검
4. annotation 자동 탐색
   - YOLO TXT
   - Pascal VOC XML
   - COCO JSON
   - bbox CSV/TSV/XLSX
5. bbox/클래스 정규화
6. exact / near duplicate 검사
7. 크기·위치·대비 분포 분석
8. duplicate-group-safe train/val/test split
9. Ultralytics YOLO 학습
10. test 평가
11. 작은 이물 / 가장자리 / 저대비 / 형상 proxy별 미탐지 분석
12. confidence threshold sweep
13. 결과 ZIP 자동 다운로드

**권장:** Colab에서 `런타임 → 런타임 유형 변경 → GPU`


In [1]:

# 0. 기본 설정
from pathlib import Path
import os, sys, json, math, random, shutil, hashlib, zipfile, warnings, base64
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORKDIR = Path("/content/kamp04")
UPLOAD_DIR = WORKDIR / "upload"
EXTRACT_DIR = WORKDIR / "extracted"
CANON_DIR = WORKDIR / "canonical_yolo"
OUT_DIR = WORKDIR / "outputs"

for d in [WORKDIR, UPLOAD_DIR, EXTRACT_DIR, CANON_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TRAINING = True
EPOCHS = 50
IMGSZ = 640
BATCH = -1
PATIENCE = 15
WORKERS = 2

RUN_NEAR_DUP_AUDIT = True
NEAR_DUP_HAMMING = 3
MAX_NEAR_DUP_IMAGES = 20000

print("WORKDIR:", WORKDIR)


WORKDIR: /content/kamp04



## 1. 대용량 ZIP 브라우저 업로드

일반 `files.upload()`는 1.7GB 파일 전체를 Python 메모리에 보관할 수 있어 불안정합니다.  
아래 셀은 브라우저의 `File.slice()`를 사용해 **4MB 단위로 런타임 디스크에 기록**합니다.


In [ ]:

# 1. Chunked browser upload
from google.colab import output

CHUNK_MB = 4
CHUNK_SIZE = CHUNK_MB * 1024 * 1024

meta = output.eval_js(r"""
(async () => {
  const input = document.createElement('input');
  input.type = 'file';
  input.accept = '.zip,application/zip';
  input.style.display = 'none';
  document.body.appendChild(input);

  const picked = new Promise(resolve => input.addEventListener('change', resolve));
  input.click();
  await picked;

  if (!input.files || input.files.length === 0) {
    input.remove();
    throw new Error('No file selected.');
  }

  window.__kamp04_file = input.files[0];
  const info = {
    name: window.__kamp04_file.name,
    size: window.__kamp04_file.size,
    type: window.__kamp04_file.type
  };
  input.remove();
  return info;
})()
""")

filename = str(meta["name"])
filesize = int(meta["size"])
dest = UPLOAD_DIR / filename

print(f"선택 파일: {filename}")
print(f"크기: {filesize/(1024**3):.3f} GB")

if dest.exists():
    dest.unlink()
dest.touch()

num_chunks = math.ceil(filesize / CHUNK_SIZE)

for i, start in enumerate(range(0, filesize, CHUNK_SIZE), 1):
    end = min(start + CHUNK_SIZE, filesize)
    js = f"""
    (async () => {{
      const blob = window.__kamp04_file.slice({start}, {end});
      const buf = await blob.arrayBuffer();
      const bytes = new Uint8Array(buf);
      let binary = '';
      const step = 32768;
      for (let j = 0; j < bytes.length; j += step) {{
        binary += String.fromCharCode(...bytes.subarray(j, Math.min(j + step, bytes.length)));
      }}
      return btoa(binary);
    }})()
    """
    b64 = output.eval_js(js)
    chunk = base64.b64decode(b64)
    with open(dest, "ab") as f:
        f.write(chunk)

    if i == 1 or i % 10 == 0 or i == num_chunks:
        print(f"\r업로드: {100*end/filesize:6.2f}% ({i}/{num_chunks})", end="")

print()
assert dest.stat().st_size == filesize, "업로드 파일 크기가 원본과 다릅니다."
print("업로드 완료:", dest)


In [ ]:

# 2. ZIP 검사 및 압축 해제
zip_path = dest
assert zipfile.is_zipfile(zip_path), "정상 ZIP으로 인식되지 않습니다."

with zipfile.ZipFile(zip_path, "r") as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"ZIP CRC 오류: {bad}")
    infos = zf.infolist()
    total_uncompressed = sum(x.file_size for x in infos)
    print("ZIP 내부 파일 수:", len(infos))
    print(f"압축 해제 예상 크기: {total_uncompressed/(1024**3):.3f} GB")

    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    zf.extractall(EXTRACT_DIR)

print("압축 해제 완료")


In [ ]:

# 3. 파일 인벤토리
all_files = [p for p in EXTRACT_DIR.rglob("*") if p.is_file()]
ext_counts = Counter(p.suffix.lower() for p in all_files)

print("전체 파일:", len(all_files))
print("\n확장자별:")
for ext, n in ext_counts.most_common(30):
    print(f"{ext or '[none]':12s} {n:8d}")

IMAGE_EXTS = {".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".webp"}
image_paths = sorted([p for p in all_files if p.suffix.lower() in IMAGE_EXTS])

print("\n이미지 수:", len(image_paths))
for p in image_paths[:10]:
    print(" -", p.relative_to(EXTRACT_DIR))

inventory = pd.DataFrame({
    "path": [str(p.relative_to(EXTRACT_DIR)) for p in all_files],
    "ext": [p.suffix.lower() for p in all_files],
    "size_bytes": [p.stat().st_size for p in all_files],
})
inventory.to_csv(OUT_DIR/"file_inventory.csv", index=False)

if not image_paths:
    raise RuntimeError("이미지를 찾지 못했습니다. file_inventory.csv를 확인하세요.")


In [ ]:

# 4. 이미지 메타데이터
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'])

import cv2
from tqdm.auto import tqdm
from IPython.display import display

img_meta = []
bad_images = []

for p in tqdm(image_paths, desc="Image metadata"):
    im = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if im is None:
        bad_images.append(str(p))
        continue
    h, w = im.shape[:2]
    channels = 1 if im.ndim == 2 else im.shape[2]
    img_meta.append({
        "image_path": str(p),
        "relative_path": str(p.relative_to(EXTRACT_DIR)),
        "stem": p.stem,
        "width": int(w),
        "height": int(h),
        "channels": int(channels),
        "dtype": str(im.dtype),
        "file_size": p.stat().st_size,
    })

img_df = pd.DataFrame(img_meta)
img_df.to_csv(OUT_DIR/"image_metadata.csv", index=False)

print("정상 이미지:", len(img_df))
print("읽기 실패:", len(bad_images))
if bad_images:
    pd.DataFrame({"bad_image": bad_images}).to_csv(OUT_DIR/"bad_images.csv", index=False)

display(img_df[["width","height","channels","file_size"]].describe())
display(img_df.groupby(["width","height"]).size().sort_values(ascending=False).head(20).to_frame("count"))



## 5. Annotation 자동 탐색

인식 가능한 형식:
- YOLO TXT
- Pascal VOC XML
- COCO JSON
- bbox CSV / TSV / XLS / XLSX

모든 annotation을 공통 bbox 스키마로 변환합니다.


In [ ]:

# 5. Annotation 자동 탐색
import xml.etree.ElementTree as ET

stem_to_images = defaultdict(list)
name_to_images = defaultdict(list)
for p in image_paths:
    stem_to_images[p.stem].append(p)
    name_to_images[p.name].append(p)

def resolve_image(name_or_stem):
    if name_or_stem is None:
        return None
    s = str(name_or_stem).replace("\\", "/")
    base = Path(s).name
    if base in name_to_images and len(name_to_images[base]) == 1:
        return name_to_images[base][0]
    stem = Path(base).stem
    cands = stem_to_images.get(stem, [])
    return cands[0] if len(cands) == 1 else None

def looks_like_yolo_txt(p):
    try:
        lines = [x.strip() for x in p.read_text(errors="ignore").splitlines() if x.strip()]
        if not lines:
            return False
        checked = 0
        for line in lines[:20]:
            parts = line.split()
            if len(parts) < 5:
                return False
            vals = [float(x) for x in parts[:5]]
            cls, xc, yc, w, h = vals
            if cls < 0 or not all(0 <= v <= 1.00001 for v in [xc,yc,w,h]):
                return False
            checked += 1
        return checked > 0
    except Exception:
        return False

records = []
detected_formats = []

# COCO JSON
for jp in [p for p in all_files if p.suffix.lower()==".json"]:
    try:
        data = json.loads(jp.read_text(errors="ignore"))
    except Exception:
        continue
    if isinstance(data, dict) and {"images","annotations"}.issubset(data.keys()):
        detected_formats.append(("COCO_JSON", str(jp)))
        id_to_img = {x["id"]:x for x in data.get("images",[])}
        cats = {x["id"]:x.get("name",str(x["id"])) for x in data.get("categories",[])}
        for ann in data.get("annotations",[]):
            imeta = id_to_img.get(ann.get("image_id"))
            if not imeta:
                continue
            ip = resolve_image(imeta.get("file_name"))
            bbox = ann.get("bbox")
            if ip is None or not bbox or len(bbox)<4:
                continue
            x,y,w,h = map(float,bbox[:4])
            cid = ann.get("category_id",0)
            records.append({
                "image_path":str(ip),"class_id_raw":cid,"class_name":cats.get(cid,str(cid)),
                "xmin":x,"ymin":y,"xmax":x+w,"ymax":y+h,
                "source_format":"COCO_JSON","source_file":str(jp)
            })

# Pascal VOC
voc_n = 0
for xp in [p for p in all_files if p.suffix.lower()==".xml"]:
    try:
        root = ET.parse(xp).getroot()
        if root.tag.lower() != "annotation":
            continue
        fn = root.findtext("filename", default=xp.stem)
        ip = resolve_image(fn) or resolve_image(xp.stem)
        if ip is None:
            continue
        objects = root.findall("object")
        if not objects:
            continue
        voc_n += 1
        for obj in objects:
            cname = obj.findtext("name", default="foreign_object")
            bb = obj.find("bndbox")
            if bb is None:
                continue
            records.append({
                "image_path":str(ip),"class_id_raw":cname,"class_name":cname,
                "xmin":float(bb.findtext("xmin")),"ymin":float(bb.findtext("ymin")),
                "xmax":float(bb.findtext("xmax")),"ymax":float(bb.findtext("ymax")),
                "source_format":"VOC_XML","source_file":str(xp)
            })
    except Exception:
        pass
if voc_n:
    detected_formats.append(("VOC_XML", f"{voc_n} files"))

# YOLO TXT
yolo_pairs = []
for tp in [p for p in all_files if p.suffix.lower()==".txt"]:
    if looks_like_yolo_txt(tp):
        ip = resolve_image(tp.stem)
        if ip is not None:
            yolo_pairs.append((tp,ip))

if yolo_pairs:
    detected_formats.append(("YOLO_TXT", f"{len(yolo_pairs)} label files"))
    meta_lookup = img_df.set_index("image_path")[["width","height"]].to_dict("index")
    for tp, ip in yolo_pairs:
        W = float(meta_lookup[str(ip)]["width"])
        H = float(meta_lookup[str(ip)]["height"])
        for line in tp.read_text(errors="ignore").splitlines():
            line=line.strip()
            if not line:
                continue
            cid,xc,yc,bw,bh = map(float,line.split()[:5])
            records.append({
                "image_path":str(ip),"class_id_raw":int(cid),"class_name":f"class_{int(cid)}",
                "xmin":(xc-bw/2)*W,"ymin":(yc-bh/2)*H,
                "xmax":(xc+bw/2)*W,"ymax":(yc+bh/2)*H,
                "source_format":"YOLO_TXT","source_file":str(tp)
            })

# bbox table
TABLE_EXTS={".csv",".tsv",".xls",".xlsx"}
IMAGE_KEYS=["image","image_path","filename","file_name","img","img_path","path"]
XMIN_KEYS=["xmin","x_min","left","x1"]
YMIN_KEYS=["ymin","y_min","top","y1"]
XMAX_KEYS=["xmax","x_max","right","x2"]
YMAX_KEYS=["ymax","y_max","bottom","y2"]
CLASS_KEYS=["class","class_name","label","category","defect","type"]

def find_col(cols, choices):
    norm={str(c).strip().lower():c for c in cols}
    for x in choices:
        if x in norm:
            return norm[x]
    return None

for fp in [p for p in all_files if p.suffix.lower() in TABLE_EXTS]:
    try:
        if fp.suffix.lower()==".csv":
            tdf=pd.read_csv(fp)
        elif fp.suffix.lower()==".tsv":
            tdf=pd.read_csv(fp,sep="\t")
        else:
            tdf=pd.read_excel(fp)
    except Exception:
        continue

    ic=find_col(tdf.columns,IMAGE_KEYS)
    x1c=find_col(tdf.columns,XMIN_KEYS)
    y1c=find_col(tdf.columns,YMIN_KEYS)
    x2c=find_col(tdf.columns,XMAX_KEYS)
    y2c=find_col(tdf.columns,YMAX_KEYS)
    cc=find_col(tdf.columns,CLASS_KEYS)

    if all(v is not None for v in [ic,x1c,y1c,x2c,y2c]):
        detected_formats.append(("BBOX_TABLE",str(fp)))
        for _,r in tdf.iterrows():
            ip=resolve_image(r[ic])
            if ip is None:
                continue
            cname=str(r[cc]) if cc is not None else "foreign_object"
            try:
                records.append({
                    "image_path":str(ip),"class_id_raw":cname,"class_name":cname,
                    "xmin":float(r[x1c]),"ymin":float(r[y1c]),
                    "xmax":float(r[x2c]),"ymax":float(r[y2c]),
                    "source_format":"BBOX_TABLE","source_file":str(fp)
                })
            except Exception:
                pass

bbox_df=pd.DataFrame(records)

print("감지된 annotation:")
for x in detected_formats:
    print(" -",x)

if len(bbox_df)==0:
    print("\n[중요] bbox annotation을 자동 인식하지 못했습니다.")
    print("file_inventory.csv와 원본 폴더 구조를 확인해야 합니다.")
    RUN_TRAINING=False
else:
    print("인식 bbox:",len(bbox_df))
    display(bbox_df.head())


In [ ]:

# 6. bbox 유효성 검증 및 class 정규화
invalid_df = pd.DataFrame()

if len(bbox_df):
    meta_map=img_df.set_index("image_path")[["width","height"]].to_dict("index")
    valid=[]
    invalid=[]

    for _,r in bbox_df.iterrows():
        m=meta_map.get(r.image_path)
        if m is None:
            d=r.to_dict(); d["reason"]="image metadata missing"; invalid.append(d); continue

        W,H=float(m["width"]),float(m["height"])
        x1,y1,x2,y2=map(float,[r.xmin,r.ymin,r.xmax,r.ymax])

        if not np.isfinite([x1,y1,x2,y2]).all():
            d=r.to_dict(); d["reason"]="non-finite bbox"; invalid.append(d); continue
        if x2<=x1 or y2<=y1:
            d=r.to_dict(); d["reason"]="non-positive bbox"; invalid.append(d); continue

        clipped=(x1<0 or y1<0 or x2>W or y2>H)
        x1=max(0,min(x1,W)); x2=max(0,min(x2,W))
        y1=max(0,min(y1,H)); y2=max(0,min(y2,H))

        if x2<=x1 or y2<=y1:
            d=r.to_dict(); d["reason"]="vanished after clip"; invalid.append(d); continue

        d=r.to_dict()
        d.update({
            "xmin":x1,"ymin":y1,"xmax":x2,"ymax":y2,
            "width_img":W,"height_img":H,"was_clipped":clipped
        })
        valid.append(d)

    bbox_df=pd.DataFrame(valid)
    invalid_df=pd.DataFrame(invalid)

    class_names=sorted(bbox_df.class_name.astype(str).unique().tolist())
    class_to_id={c:i for i,c in enumerate(class_names)}
    bbox_df["class_id"]=bbox_df.class_name.astype(str).map(class_to_id)

    bbox_df["bw"]=bbox_df.xmax-bbox_df.xmin
    bbox_df["bh"]=bbox_df.ymax-bbox_df.ymin
    bbox_df["area_ratio"]=(bbox_df.bw*bbox_df.bh)/(bbox_df.width_img*bbox_df.height_img)
    bbox_df["xc_norm"]=((bbox_df.xmin+bbox_df.xmax)/2)/bbox_df.width_img
    bbox_df["yc_norm"]=((bbox_df.ymin+bbox_df.ymax)/2)/bbox_df.height_img
    bbox_df["edge_distance"]=np.minimum.reduce([
        bbox_df.xc_norm,1-bbox_df.xc_norm,bbox_df.yc_norm,1-bbox_df.yc_norm
    ])
    bbox_df["aspect_ratio"]=bbox_df.bw/bbox_df.bh

    bbox_df.to_csv(OUT_DIR/"canonical_bboxes.csv",index=False)
    invalid_df.to_csv(OUT_DIR/"invalid_bboxes.csv",index=False)

    print("유효 bbox:",len(bbox_df))
    print("무효 bbox:",len(invalid_df))
    print("class map:",class_to_id)
    display(bbox_df.groupby("class_name").size().sort_values(ascending=False).to_frame("boxes"))

    annotated=set(bbox_df.image_path)
    print("전체 이미지:",len(img_df))
    print("bbox 있음:",len(annotated))
    print("bbox 없음:",len(img_df)-len(annotated))


In [ ]:

# 7. exact / near duplicate 검사
import itertools

def sha1_file(path, chunk=1024*1024):
    h=hashlib.sha1()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

hash_rows=[]
for p in tqdm(image_paths,desc="SHA1"):
    hash_rows.append({"image_path":str(p),"sha1":sha1_file(p)})
hash_df=pd.DataFrame(hash_rows)
hash_df.to_csv(OUT_DIR/"image_hashes.csv",index=False)

dup_counts=hash_df.groupby("sha1").size().sort_values(ascending=False)
exact_dup_groups=dup_counts[dup_counts>1]
print("Exact duplicate groups:",len(exact_dup_groups))
print("Images in exact duplicate groups:",int(exact_dup_groups.sum()) if len(exact_dup_groups) else 0)

hash_df[hash_df.sha1.isin(exact_dup_groups.index)].sort_values("sha1").to_csv(
    OUT_DIR/"exact_duplicate_images.csv",index=False
)

def dhash64(path,hash_size=8):
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    if im is None:
        return None
    small=cv2.resize(im,(hash_size+1,hash_size),interpolation=cv2.INTER_AREA)
    diff=small[:,1:]>small[:,:-1]
    bits=0
    for v in diff.flatten():
        bits=(bits<<1)|int(v)
    return bits

def hamming64(a,b):
    return int((a^b).bit_count())

near_pairs=[]
if RUN_NEAR_DUP_AUDIT:
    subset=image_paths[:MAX_NEAR_DUP_IMAGES]
    dh=[]
    for p in tqdm(subset,desc="dHash"):
        v=dhash64(p)
        if v is not None:
            dh.append((p,v))

    buckets=defaultdict(list)
    for p,v in dh:
        buckets[(v>>48)&0xFFFF].append((p,v))

    for bucket in buckets.values():
        if len(bucket)<2:
            continue
        for (p1,h1),(p2,h2) in itertools.combinations(bucket,2):
            d=hamming64(h1,h2)
            if d<=NEAR_DUP_HAMMING:
                near_pairs.append({"image1":str(p1),"image2":str(p2),"dhash_distance":d})

near_df=pd.DataFrame(near_pairs)
near_df.to_csv(OUT_DIR/"near_duplicate_candidates.csv",index=False)
print("Near duplicate candidate pairs:",len(near_df))


In [ ]:

# 8. X-ray bbox 조건 feature + EDA
import matplotlib.pyplot as plt

if len(bbox_df):
    def bbox_contrast(row,pad_ratio=0.5):
        im=cv2.imread(row.image_path,cv2.IMREAD_GRAYSCALE)
        if im is None:
            return np.nan
        H,W=im.shape[:2]
        x1,y1,x2,y2=map(int,[row.xmin,row.ymin,row.xmax,row.ymax])
        bw,bh=max(1,x2-x1),max(1,y2-y1)
        px,py=int(bw*pad_ratio),int(bh*pad_ratio)
        rx1,ry1=max(0,x1-px),max(0,y1-py)
        rx2,ry2=min(W,x2+px),min(H,y2+py)

        roi=im[y1:y2,x1:x2].astype(np.float32)
        ring=im[ry1:ry2,rx1:rx2].astype(np.float32)
        if roi.size==0 or ring.size==0:
            return np.nan

        mask=np.ones(ring.shape,dtype=bool)
        ax1,ay1=x1-rx1,y1-ry1
        ax2,ay2=x2-rx1,y2-ry1
        mask[ay1:ay2,ax1:ax2]=False
        bg=ring[mask]
        if bg.size==0:
            return np.nan
        return abs(float(roi.mean())-float(bg.mean()))/255.0

    bbox_df["contrast_proxy"]=[
        bbox_contrast(r) for _,r in tqdm(bbox_df.iterrows(),total=len(bbox_df),desc="Contrast")
    ]
    bbox_df.to_csv(OUT_DIR/"canonical_bboxes_with_features.csv",index=False)

    print("Object size:")
    display(bbox_df.area_ratio.describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]))
    print("Edge distance:")
    display(bbox_df.edge_distance.describe())
    print("Contrast proxy:")
    display(bbox_df.contrast_proxy.describe())

    plt.figure(figsize=(7,4))
    cap=bbox_df.area_ratio.quantile(.99)
    plt.hist(bbox_df.area_ratio.clip(upper=cap),bins=50)
    plt.xlabel("BBox area / image area")
    plt.ylabel("Count")
    plt.title("Foreign-object size distribution")
    plt.tight_layout()
    plt.savefig(OUT_DIR/"eda_object_size.png",dpi=160)
    plt.show()

    plt.figure(figsize=(7,4))
    plt.hist(bbox_df.edge_distance,bins=40)
    plt.xlabel("Distance of bbox center to nearest image edge")
    plt.ylabel("Count")
    plt.title("Object position distribution")
    plt.tight_layout()
    plt.savefig(OUT_DIR/"eda_edge_distance.png",dpi=160)
    plt.show()

    vals=bbox_df.contrast_proxy.dropna()
    plt.figure(figsize=(7,4))
    plt.hist(vals,bins=40)
    plt.xlabel("ROI-background contrast proxy")
    plt.ylabel("Count")
    plt.title("X-ray contrast distribution")
    plt.tight_layout()
    plt.savefig(OUT_DIR/"eda_contrast.png",dpi=160)
    plt.show()


In [ ]:

# 9. Exact-duplicate-group-safe split
if RUN_TRAINING and len(bbox_df):
    base=img_df[["image_path","width","height"]].copy()
    base=base.merge(hash_df,on="image_path",how="left")
    positive_set=set(bbox_df.image_path)
    base["has_object"]=base.image_path.isin(positive_set).astype(int)

    group_df=base.groupby("sha1").agg(
        n=("image_path","size"),
        has_object=("has_object","max")
    ).reset_index()

    def split_groups(gdf,seed):
        rng=np.random.RandomState(seed)
        groups=gdf.sha1.to_numpy().copy()
        rng.shuffle(groups)
        n=len(groups)
        n_train=int(round(n*.70))
        n_val=int(round(n*.15))
        return set(groups[:n_train]),set(groups[n_train:n_train+n_val]),set(groups[n_train+n_val:])

    train_groups,val_groups,test_groups=set(),set(),set()
    for label,sub in group_df.groupby("has_object"):
        tr,va,te=split_groups(sub,SEED+int(label))
        train_groups|=tr; val_groups|=va; test_groups|=te

    def assign_split(h):
        if h in train_groups: return "train"
        if h in val_groups: return "val"
        return "test"

    base["split"]=base.sha1.map(assign_split)

    display(base.groupby(["split","has_object"]).size().unstack(fill_value=0))
    assert base.groupby("sha1").split.nunique().max()==1
    print("Exact duplicate leakage: 없음")

    base.to_csv(OUT_DIR/"image_splits.csv",index=False)


In [ ]:

# 10. 표준 YOLO dataset 생성
import yaml

if RUN_TRAINING and len(bbox_df):
    if CANON_DIR.exists():
        shutil.rmtree(CANON_DIR)

    for split in ["train","val","test"]:
        (CANON_DIR/"images"/split).mkdir(parents=True,exist_ok=True)
        (CANON_DIR/"labels"/split).mkdir(parents=True,exist_ok=True)

    img_to_out={}

    for _,r in tqdm(base.iterrows(),total=len(base),desc="Build YOLO dataset"):
        src=Path(r.image_path)
        split=r.split
        out_name=f"{r.sha1[:10]}_{src.name}"
        out_img=CANON_DIR/"images"/split/out_name

        try:
            os.link(src,out_img)
        except Exception:
            shutil.copy2(src,out_img)

        img_to_out[str(src)]=out_img

        boxes=bbox_df[bbox_df.image_path==str(src)]
        label_path=CANON_DIR/"labels"/split/(Path(out_name).stem+".txt")
        lines=[]
        W,H=float(r.width),float(r.height)

        for _,b in boxes.iterrows():
            xc=((b.xmin+b.xmax)/2)/W
            yc=((b.ymin+b.ymax)/2)/H
            bw=(b.xmax-b.xmin)/W
            bh=(b.ymax-b.ymin)/H
            if all(0<=v<=1 for v in [xc,yc,bw,bh]) and bw>0 and bh>0:
                lines.append(f"{int(b.class_id)} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}")

        label_path.write_text("\n".join(lines))

    yaml_data={
        "path":str(CANON_DIR),
        "train":"images/train",
        "val":"images/val",
        "test":"images/test",
        "names":{int(v):k for k,v in class_to_id.items()}
    }
    yaml_path=CANON_DIR/"dataset.yaml"
    yaml_path.write_text(yaml.safe_dump(yaml_data,sort_keys=False,allow_unicode=True))

    print(yaml_path.read_text())


In [ ]:
# 11. Ultralytics 설치 + GPU 확인
if RUN_TRAINING and len(bbox_df):
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics'])

    import torch, ultralytics
    from ultralytics import YOLO

    print("Ultralytics:", ultralytics.__version__)
    print("PyTorch:", torch.__version__)
    print("CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("[주의] GPU 런타임이 아닙니다.")


In [ ]:

# 12. Pretrained detector 자동 선택 + 학습
if RUN_TRAINING and len(bbox_df):
    model=None
    model_name=None

    for cand in ["yolo26n.pt","yolo11n.pt","yolov8n.pt"]:
        try:
            print("시도:",cand)
            model=YOLO(cand)
            model_name=cand
            print("사용 모델:",cand)
            break
        except Exception as e:
            print("실패:",str(e)[:200])

    if model is None:
        raise RuntimeError("Ultralytics pretrained detector를 불러오지 못했습니다.")

    train_result=model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        workers=WORKERS,
        seed=SEED,
        deterministic=True,
        project=str(OUT_DIR/"training"),
        name="xray_detector",
        exist_ok=True,
        plots=True,
        cache=False,
        verbose=True
    )

    best_pt=OUT_DIR/"training"/"xray_detector"/"weights"/"best.pt"
    assert best_pt.exists(),f"best.pt 없음: {best_pt}"
    print("Best model:",best_pt)


In [ ]:

# 13. 표준 test 평가
if RUN_TRAINING and len(bbox_df):
    best_model=YOLO(str(best_pt))

    metrics=best_model.val(
        data=str(yaml_path),
        split="test",
        imgsz=IMGSZ,
        conf=0.001,
        iou=0.5,
        plots=True,
        project=str(OUT_DIR/"validation"),
        name="test",
        exist_ok=True
    )

    summary={
        "model":model_name,
        "map50_95":float(metrics.box.map),
        "map50":float(metrics.box.map50),
        "map75":float(metrics.box.map75),
        "precision":float(metrics.box.mp),
        "recall":float(metrics.box.mr)
    }
    print(json.dumps(summary,indent=2))
    pd.DataFrame([summary]).to_csv(OUT_DIR/"test_metrics.csv",index=False)


In [ ]:

# 14. 객체별 low-confidence prediction + GT matching
def iou_xyxy(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    bb=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    den=aa+bb-inter
    return inter/den if den>0 else 0.0

if RUN_TRAINING and len(bbox_df):
    test_base=base[base.split=="test"].copy()
    test_paths=[str(img_to_out[p]) for p in test_base.image_path]

    preds=best_model.predict(
        source=test_paths,
        imgsz=IMGSZ,
        conf=0.001,
        iou=0.7,
        verbose=False,
        stream=False
    )

    out_to_orig={str(v):k for k,v in img_to_out.items()}
    pred_by_orig=defaultdict(list)

    for res in preds:
        op=str(Path(res.path))
        orig=out_to_orig.get(op)
        if orig is None:
            matches=[k for k,v in img_to_out.items() if Path(v).name==Path(op).name]
            orig=matches[0] if len(matches)==1 else None
        if orig is None or res.boxes is None:
            continue

        for xyxy,conf,cls in zip(
            res.boxes.xyxy.cpu().numpy(),
            res.boxes.conf.cpu().numpy(),
            res.boxes.cls.cpu().numpy()
        ):
            pred_by_orig[orig].append({
                "box":xyxy.astype(float).tolist(),
                "conf":float(conf),
                "class_id":int(cls)
            })

    gt_test=bbox_df[bbox_df.image_path.isin(set(test_base.image_path))].copy()

    match_rows=[]
    for _,gt in tqdm(gt_test.iterrows(),total=len(gt_test),desc="GT matching"):
        preds_img=pred_by_orig.get(gt.image_path,[])
        gtbox=[gt.xmin,gt.ymin,gt.xmax,gt.ymax]
        best_iou=0.0
        best_conf=0.0

        for pr in preds_img:
            if int(pr["class_id"])!=int(gt.class_id):
                continue
            iv=iou_xyxy(gtbox,pr["box"])
            if iv>best_iou:
                best_iou=iv
                best_conf=pr["conf"]

        d=gt.to_dict()
        d.update({
            "best_iou":best_iou,
            "best_conf":best_conf,
            "detected_at_005":int(best_iou>=.5 and best_conf>=.05),
            "detected_at_025":int(best_iou>=.5 and best_conf>=.25)
        })
        match_rows.append(d)

    match_df=pd.DataFrame(match_rows)
    match_df.to_csv(OUT_DIR/"test_gt_detection_matches.csv",index=False)

    print("Test GT objects:",len(match_df))
    if len(match_df):
        print("Recall conf>=.05:",match_df.detected_at_005.mean())
        print("Recall conf>=.25:",match_df.detected_at_025.mean())


In [ ]:

# 15. 조건별 미탐지 분석
if RUN_TRAINING and len(bbox_df) and len(match_df):
    ana=match_df.copy()

    def qbin(s,labels):
        try:
            return pd.qcut(s,q=len(labels),labels=labels,duplicates="drop")
        except Exception:
            return pd.Series(["all"]*len(s),index=s.index)

    ana["size_group"]=qbin(ana.area_ratio,["small","medium","large"])
    ana["position_group"]=np.where(ana.edge_distance<.15,"edge","non_edge")
    fill_contrast=ana.contrast_proxy.fillna(ana.contrast_proxy.median())
    ana["contrast_group"]=qbin(fill_contrast,["low","medium","high"])
    ana["shape_proxy"]=np.where(
        (ana.aspect_ratio>=2)|(ana.aspect_ratio<=.5),
        "elongated_bbox","compact_bbox"
    )

    tables=[]
    for col in ["size_group","position_group","contrast_group","shape_proxy","class_name"]:
        t=ana.groupby(col,observed=True).agg(
            n=("detected_at_025","size"),
            recall_conf025=("detected_at_025","mean"),
            mean_best_conf=("best_conf","mean"),
            mean_best_iou=("best_iou","mean")
        ).reset_index()
        t.insert(0,"condition",col)
        t=t.rename(columns={col:"group"})
        tables.append(t)

    condition_df=pd.concat(tables,ignore_index=True)
    condition_df.to_csv(OUT_DIR/"condition_recall_analysis.csv",index=False)
    display(condition_df)

    for col in ["size_group","position_group","contrast_group","shape_proxy"]:
        t=ana.groupby(col,observed=True).detected_at_025.mean()
        plt.figure(figsize=(6,4))
        t.plot(kind="bar")
        plt.ylim(0,1)
        plt.ylabel("Recall")
        plt.title(f"Recall by {col}")
        plt.tight_layout()
        plt.savefig(OUT_DIR/f"recall_by_{col}.png",dpi=160)
        plt.show()


In [ ]:

# 16. Confidence threshold sweep
if RUN_TRAINING and len(bbox_df):
    thresholds=np.round(np.arange(.01,.91,.02),2)
    test_orig_paths=test_base.image_path.tolist()
    sweep=[]

    for th in thresholds:
        TP=FP=FN=0

        for orig in test_orig_paths:
            gt_img=gt_test[gt_test.image_path==orig]
            pr_img=[p for p in pred_by_orig.get(orig,[]) if p["conf"]>=th]

            gt_used=set()
            pr_used=set()
            pairs=[]

            for gi,(_,gt) in enumerate(gt_img.iterrows()):
                for pi,pr in enumerate(pr_img):
                    if int(pr["class_id"])!=int(gt.class_id):
                        continue
                    iv=iou_xyxy([gt.xmin,gt.ymin,gt.xmax,gt.ymax],pr["box"])
                    if iv>=.5:
                        pairs.append((iv,gi,pi))

            for iv,gi,pi in sorted(pairs,reverse=True):
                if gi not in gt_used and pi not in pr_used:
                    gt_used.add(gi)
                    pr_used.add(pi)

            TP+=len(gt_used)
            FN+=len(gt_img)-len(gt_used)
            FP+=len(pr_img)-len(pr_used)

        precision=TP/(TP+FP) if TP+FP else 0.0
        recall=TP/(TP+FN) if TP+FN else 0.0
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
        beta=2
        f2=(1+beta**2)*precision*recall/(beta**2*precision+recall) if precision+recall else 0.0

        sweep.append({
            "threshold":th,"TP":TP,"FP":FP,"FN":FN,
            "precision":precision,"recall":recall,"f1":f1,"f2":f2
        })

    sweep_df=pd.DataFrame(sweep)
    sweep_df.to_csv(OUT_DIR/"confidence_threshold_sweep.csv",index=False)

    best_f2=sweep_df.loc[sweep_df.f2.idxmax()]
    print("Max-F2 high-sensitivity candidate:")
    display(best_f2.to_frame().T)

    plt.figure(figsize=(7,4))
    plt.plot(sweep_df.threshold,sweep_df.precision,label="Precision")
    plt.plot(sweep_df.threshold,sweep_df.recall,label="Recall")
    plt.plot(sweep_df.threshold,sweep_df.f2,label="F2")
    plt.xlabel("Confidence threshold")
    plt.ylabel("Score")
    plt.ylim(0,1)
    plt.legend()
    plt.title("Threshold trade-off")
    plt.tight_layout()
    plt.savefig(OUT_DIR/"threshold_tradeoff.png",dpi=160)
    plt.show()


In [ ]:

# 17. 데이터 건전성 요약
health={
    "n_files":len(all_files),
    "n_images":len(img_df),
    "n_bad_images":len(bad_images),
    "annotation_formats":detected_formats,
    "n_valid_boxes":int(len(bbox_df)),
    "n_invalid_boxes":int(len(invalid_df)),
    "n_annotated_images":int(bbox_df.image_path.nunique()) if len(bbox_df) else 0,
    "n_exact_duplicate_groups":int(len(exact_dup_groups)),
    "n_images_in_exact_duplicate_groups":int(exact_dup_groups.sum()) if len(exact_dup_groups) else 0,
    "n_near_duplicate_candidate_pairs":int(len(near_df))
}

if len(bbox_df):
    health.update({
        "n_classes":int(bbox_df.class_name.nunique()),
        "median_object_area_ratio":float(bbox_df.area_ratio.median()),
        "p10_object_area_ratio":float(bbox_df.area_ratio.quantile(.10)),
        "edge_object_fraction":float((bbox_df.edge_distance<.15).mean()),
        "median_contrast_proxy":float(bbox_df.contrast_proxy.median())
    })

(OUT_DIR/"dataset_health_summary.json").write_text(
    json.dumps(health,indent=2,ensure_ascii=False,default=str)
)

print(json.dumps(health,indent=2,ensure_ascii=False,default=str))
print("핵심 판정 포인트:")
print("1) bbox annotation이 충분한가")
print("2) small/edge/low-contrast 이물이 충분한가")
print("3) exact/near duplicate 비율이 높은가")
print("4) duplicate split leakage는 차단되었는가")
print("5) 조건별 recall 차이가 실제로 존재하는가")


In [ ]:

# 18. 결과 ZIP 생성 + 다운로드
if RUN_TRAINING and "best_pt" in globals() and Path(best_pt).exists():
    shutil.copy2(best_pt,OUT_DIR/"best.pt")
if "yaml_path" in globals() and Path(yaml_path).exists():
    shutil.copy2(yaml_path,OUT_DIR/"dataset.yaml")

zip_out=shutil.make_archive("/content/KAMP04_xray_analysis_results","zip",OUT_DIR)
print("결과 ZIP:",zip_out)

from google.colab import files
files.download(zip_out)



## 실행 후 보내줄 파일

마지막에 자동 다운로드되는:

**`KAMP04_xray_analysis_results.zip`**

을 여기 올려주면 됩니다.

그 결과로:
- ④ 데이터가 실제로 깨끗한지
- 작은 이물 / 가장자리 / 저대비 이물 분석이 가능한지
- duplicate 때문에 성능이 부풀려지는지
- ③과 ④ 중 어느 주제가 더 유의미한지

까지 다시 판단할 수 있습니다.
